In [1]:
import os
from pathlib import Path

# 1) Đặt JAVA_HOME TRƯỚC khi import py_vncorenlp
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = (
    r"C:\Program Files\Java\jdk-17\bin\server" + os.pathsep
    + r"C:\Program Files\Java\jdk-17\bin" + os.pathsep
    + os.environ["PATH"]
)

import py_vncorenlp

# 2) Thư mục có VnCoreNLP-1.2.jar và thư mục models/ (đúng như bạn đã kiểm tra)
MODEL_DIR = Path(r"D:\NamCuoi\Code\Project\PhatHienNoiDungDocHai\Code\models\VnCoreNLP")

cwd = os.getcwd()  # py_vncorenlp tự đổi cwd, nên lưu lại để trả về
segmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(MODEL_DIR))
os.chdir(cwd)

text = "Chúng tôi đang nghiên cứu xử lý ngôn ngữ tự nhiên."
print(segmenter.word_segment(text))

['Chúng_tôi đang nghiên_cứu xử_lý ngôn_ngữ_tự_nhiên .']


In [2]:
import pandas as pd 
df_train = pd.read_csv("D:/NamCuoi/Code/Project/PhatHienNoiDungDocHai/Data/ViCTSD-main/ViCTSD_train.csv")
df_val = pd.read_csv("D:/NamCuoi/Code/Project/PhatHienNoiDungDocHai/Data/ViCTSD-main/ViCTSD_valid.csv")
df_test = pd.read_csv("D:/NamCuoi/Code/Project/PhatHienNoiDungDocHai/Data/ViCTSD-main/ViCTSD_test.csv")

In [3]:
print(df_train.shape)
print(df_val.shape)
print(df_test.shape)


(7000, 6)
(2000, 6)
(1000, 6)


In [4]:
def segment_comments(df):
    df = df.copy()
    # Xử lý giá trị thiếu và đảm bảo đầu vào là chuỗi
    comments = df["Comment"].fillna("").astype(str)
    df["Comment_segmented"] = [
        segmenter.word_segment(comment)
        for comment in comments
    ]
    return df


train_df = segment_comments(df_train)
valid_df = segment_comments(df_val)
test_df = segment_comments(df_test)

print("Đã phân đoạn xong!")
print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

display(train_df[["Comment", "Comment_segmented", "Toxicity"]].head())

Đã phân đoạn xong!
Train: (7000, 7)
Validation: (2000, 7)
Test: (1000, 7)


,Comment,Comment_segmented,Toxicity
0,Thật tuyệt vời...!!!,"[Thật tuyệt_vời ..., !, !, !]",0
1,"mỹ đã tuột dốc quá nhiều rồi, giờ muốn vực dậy...","[mỹ đã tuột_dốc quá nhiều rồi , giờ muốn vực d...",0
2,tôi thấy người lái xe hơi bấm còi mới là người...,[tôi thấy người lái_xe hơi bấm còi mới là ngườ...,1
3,Coi dịch là giặc. Đã mang tên đó mà xâm nhập V...,"[Coi dịch là giặc ., Đã mang tên đó mà xâm_nhậ...",0
4,Thương các bé quá! Các con còn quá nhỏ mà đã p...,"[Thương các bé quá !, Các con còn quá nhỏ mà đ...",0


In [5]:

from pathlib import Path
output_dir = Path("data_segmented")
output_dir.mkdir(parents=True, exist_ok=True)
# Lưu ba tập dữ liệu
train_df.to_csv(output_dir / "train.csv", index=False, encoding="utf-8-sig")
valid_df.to_csv(output_dir / "valid.csv", index=False, encoding="utf-8-sig")
test_df.to_csv(output_dir / "test.csv", index=False, encoding="utf-8-sig")

print("Đã lưu dữ liệu vào:", output_dir.resolve())


Đã lưu dữ liệu vào: D:\NamCuoi\Code\Project\PhatHienNoiDungDocHai\Code\data_segmented


In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

In [7]:
#test trên 1 cmt
text = train_df["Comment_segmented"].iloc[0]
encoded = tokenizer(text,add_special_tokens=True)
print("Bình luận:", text)

print("Token:", tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))
print("Token IDs:", encoded["input_ids"])
print("Độ dài:", len(encoded["input_ids"]))

Bình luận: ['Thật tuyệt_vời ...', '!', '!', '!']
Token: ['<s>', 'Thật', 'tuyệt_vời', '...', '</s>']
Token IDs: [[0, 3395, 1819, 135, 2], [0, 381, 2], [0, 381, 2], [0, 381, 2]]
Độ dài: 4
